# Loan Default Prediction — Business Recommendations

**Audience:** Credit Risk & Portfolio Management teams

**Purpose:** Translate model findings into actionable recommendations for reducing credit losses while maintaining loan volume.

## Executive Summary

We built a gradient-boosted classifier to predict loan default probability at the point of application. The model achieves a **ROC-AUC of ~0.77** on held-out data — significantly better than the random baseline (0.50) and the logistic regression benchmark (~0.69).

By lowering the classification threshold from 0.5 to the optimised value, we increase **recall on the default class from ~55% to ~75%**, meaning the model catches 3 in 4 loans that would have defaulted — enabling proactive intervention before funds are disbursed.

---

## 1. Top Default Risk Drivers

Based on SHAP analysis, the five strongest predictors of default are:

| Rank | Feature | Direction | Interpretation |
|---|---|---|---|
| 1 | `EXT_SOURCE_2` | Lower → higher risk | Third-party credit score is the single strongest signal |
| 2 | `EXT_SOURCE_3` | Lower → higher risk | Second external credit score reinforces the above |
| 3 | `EXT_SOURCE_1` | Lower → higher risk | Combined, the three scores explain the majority of model variance |
| 4 | `AGE_YEARS` | Younger → higher risk | Applicants under 35 default at nearly 2x the rate of those over 55 |
| 5 | `CREDIT_INCOME_RATIO` | Higher → higher risk | Borrowing more than ~4x annual income significantly elevates risk |

## 2. Risk Segmentation

Using predicted default probability, applicants can be segmented into three tiers for differentiated treatment:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Illustrative risk tier distribution based on model output
tiers = pd.DataFrame({
    'Tier': ['Low Risk\n(p < 0.15)', 'Medium Risk\n(0.15 ≤ p < 0.35)', 'High Risk\n(p ≥ 0.35)'],
    'Share of Applicants (%)': [62, 27, 11],
    'Est. Default Rate (%)': [3.2, 12.5, 38.4],
    'Recommended Action': [
        'Auto-approve at standard terms',
        'Manual review or adjusted terms',
        'Decline or require collateral'
    ],
    'Color': ['#43A047', '#FB8C00', '#E53935']
})

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Applicant share
bars = axes[0].bar(tiers['Tier'], tiers['Share of Applicants (%)'], color=tiers['Color'])
axes[0].set_ylabel('Share of Applicants (%)')
axes[0].set_title('Applicant Distribution by Risk Tier', fontweight='bold')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height()}%', ha='center', fontweight='bold')

# Default rate per tier
bars2 = axes[1].bar(tiers['Tier'], tiers['Est. Default Rate (%)'], color=tiers['Color'])
axes[1].set_ylabel('Estimated Default Rate (%)')
axes[1].set_title('Default Rate by Risk Tier', fontweight='bold')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{bar.get_height()}%', ha='center', fontweight='bold')

plt.suptitle('Risk Segmentation Framework', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(tiers[['Tier', 'Recommended Action']].to_string(index=False))

## 3. Business Impact Estimation

Assuming:
- Average loan size: **₱200,000**
- Average loss given default (LGD): **60%** (of principal)
- Monthly applications: **10,000**

In [ ]:
avg_loan = 200_000
lgd = 0.60
monthly_apps = 10_000
overall_default_rate = 0.082

# Without model — all applications approved
expected_defaults_no_model = monthly_apps * overall_default_rate
expected_loss_no_model = expected_defaults_no_model * avg_loan * lgd

# With model — high-risk tier declined (11% of apps, ~38% default rate)
high_risk_apps = monthly_apps * 0.11
defaults_prevented = high_risk_apps * 0.384
loss_prevented = defaults_prevented * avg_loan * lgd

# Volume impact
volume_reduction = high_risk_apps / monthly_apps

print('=== Monthly Impact Estimate ===')
print(f'Expected defaults without model:   {expected_defaults_no_model:.0f}')
print(f'Estimated defaults prevented:      {defaults_prevented:.0f}')
print(f'Estimated credit loss prevented:   ₱{loss_prevented:,.0f}')
print(f'Application volume reduction:      {volume_reduction:.1%}')
print(f'\nAnnualised loss prevention estimate: ₱{loss_prevented*12:,.0f}')

## 4. Recommendations

### R1 — Deploy a Three-Tier Approval Framework
Integrate the model's predicted probability into the loan decisioning workflow. Auto-approve low-risk applicants, route medium-risk to manual review, and decline or require collateral for high-risk applicants. This reduces manual review volume by ~62% while concentrating human judgment where it matters most.

### R2 — Prioritise External Credit Score Completeness
The `EXT_SOURCE` features collectively account for the largest share of predictive power. Data completeness for these scores is ~65–80%. Partnering with additional credit bureaus or improving consent capture rates would directly improve model accuracy.

### R3 — Apply Risk-Based Pricing for Medium-Risk Applicants
Rather than declining borderline applicants outright, apply risk-adjusted interest rates or lower credit limits for the medium-risk tier. This preserves loan volume while pricing risk appropriately.

### R4 — Monitor for Model Drift
Credit behaviour shifts over time (e.g., economic downturns, policy changes). Schedule quarterly retraining and track PSI (Population Stability Index) on the score distribution. Set alerts if PSI > 0.2.

### R5 — Expand to Behavioural Data
The current model uses only application-level data. Incorporating repayment behaviour on existing products (e.g., credit card utilisation, missed payments) would materially improve recall on first-time borrowers — the most uncertain segment.

## 5. Model Limitations

| Limitation | Mitigation |
|---|---|
| Trained only on application data | Add behavioural/transactional features in v2 |
| Static threshold — not tailored by product type | Train separate models per loan product |
| Class imbalance handled via weights, not SMOTE | Test SMOTE in a follow-up experiment |
| No fairness audit conducted | Evaluate demographic parity across gender and age groups before production |

---

## 6. Next Steps

1. **Validation** — test on a holdout set from a different time period to check for temporal drift
2. **Fairness audit** — check model performance across demographic subgroups
3. **Feature expansion** — incorporate bureau sub-tables (`bureau.csv`, `previous_application.csv`)
4. **Production packaging** — wrap model in a scoring API using FastAPI or Flask
5. **A/B test** — shadow-deploy the model alongside the current approval process to measure real-world lift